## Module 3: Computational Drug Discovery Project 

## Project Goal:

You are working as a Quantitative Systems Pharmacologist at the UVA Manning Institute for Biotechnology. Your overall goal is to help develop a GLP1 agonist drug for weight loss, branded Cavalitide, that will compete with semaglutide (Ozempic or Wegovy). Your project is composed of three tasks:
* Task 1: Fit a receptor binding (pharmacodynamics) model for Cavalitide
* Task 2: Simulate the pharmacokinetics of Cavalitide treatment
* Task 3: Simulate a combined pharmacokinetics-pharmacodynamics (PKPD) model that predicts GLP1 receptor activity during treatment.
  
The below sections should be addressed in your Project Report as a Jupyter notebook, based on Module3_Project_Template.ipynb. All groups submit a Project Report, only those in the Sullivan Company have Oral Presentations (**all due 11/17**).

## Disease Background: 

Obesity is a multifactorial condition involving the accumulation of too much body fat. Diagnosis of the condition is from a calculated Body Mass Index (BMI) greater than 30.0. Factors that can contribute to the develop of the disease include unhealthy diet, inactivity, certain disease (such as Cushing syndrome or hypothyroidism), low social and economic status, lack of sleep, etc. Treatment includes increased physical activity, reduction in caloric intake and GLP-1 agonists.(Mayo Clinic, 2023) Due to increasingly high demands of modern life and the sedantry nature of many modern middle class jobs, many patients with obesity struggle to find the time and money to fuel a better diet and increased physical activity. Therefore, medications such as GLP-1 agonists, which can increased satiety, promise a more convenient and realistic way for obesity patients to reduce caloric intake.

## Task 1: Fit a receptor binding model for Cavalitide:

Your colleague in Lead Discovery has identified a small molecule, Cavalitide, and performed experiments testing its binding to the GLP1 receptor (below, and available as cavalitide_data.csv). You now aim to develop a pharmacodynamics model of how GLP1 and Cavalitide can each bind and activvate the GLP1 receptor.
a)	Adapt your previous codes for modeling semaglutide (fit_semaglutide.py, semaglutide_data.csv) to fit the Cavalitide binding data:

#fit_cavalitide_GLP1R.py
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def Fact(L1, L2, Kd1, Kd2):
    """Fractional receptor activation with two agonists L1 and L2"""
    return (Kd2*L1 + Kd1*L2) / (Kd1*Kd2 + Kd2*L1 + Kd1*L2)

# Parameters [nM]
Kd1 = 0.17      # L1, GLP1
Kd2_sema = 0.38 # semaglutide (reference)
L1 = 0.015      # GLP1 (fasting)
L2 = np.logspace(-3, 2, 500)

# Load the data 
data = np.loadtxt("cavalitide_data.csv", delimiter=",", skiprows=1)
L2_loaded, Fact_loaded = data[:, 0], data[:, 1]

# Approach 1: brute force
Kd2_candidates = np.logspace(-3, 3, 2000)
SSEs = [np.sum((Fact_loaded - Fact(L1, L2_loaded, Kd1, k))**2) for k in Kd2_candidates]
i = int(np.argmin(SSEs))
Kd2_brute, SSE_brute = Kd2_candidates[i], SSEs[i]
print(f"Brute-force Kd2 = {Kd2_brute:.4f} nM (SSE = {SSE_brute:.5f})")

# Approach 2: curve_fit
popt, pcov = curve_fit(lambda L2, Kd2: Fact(L1, L2, Kd1, Kd2),
                       L2_loaded, Fact_loaded, p0=[Kd2_brute], bounds=(0, np.inf))
Kd2_fit = popt[0]
SSE_fit = np.sum((Fact_loaded - Fact(L1, L2_loaded, Kd1, Kd2_fit))**2)
print(f"Fitted Kd2 = {Kd2_fit:.4f} +/- {np.sqrt(pcov[0,0]):.4f} nM (SSE = {SSE_fit:.5f})")

plt.figure(figsize=(7, 4.5))
plt.semilogx(L2_loaded, Fact_loaded, "ko", label="experimental data")
plt.semilogx(L2, Fact(L1, L2, Kd1, Kd2_brute), ":", lw=2.5,
             label=f"brute force (Kd = {Kd2_brute:.3f} nM)")
plt.semilogx(L2, Fact(L1, L2, Kd1, Kd2_fit), "--", lw=2,
             label=f"curve_fit (Kd = {Kd2_fit:.3f} nM)")
plt.semilogx(L2, Fact(L1, L2, Kd1, Kd2_sema), "-", lw=1.2, alpha=0.5,
             label=f"semaglutide ref. (Kd = {Kd2_sema} nM)")
plt.ylim(-0.1, 1.1)
plt.xlabel("[Cavalitide] (nM)")
plt.ylabel("Fraction active receptors (Fact)")
plt.title("GLP1R activation by Cavalitide with GLP1 present")
plt.legend(loc="upper left")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("cavalitide_fit.png", dpi=150)
plt.show()

b)	Show a plot of [Cavalitide] vs. Fraction active receptors (Fact) demonstrating a good fit.

c)	Provide the fitted value of Kd for Cavalitide. Interpret the Kd of Cavalitide versus that of semaglutide, commenting on advantages/disadvantages:

 Fitted Kd for Cavalitide: 0.077 ± 0.005 nM 
 Compared to semaglutide (Kd=0.38mM), the Kd value is much lower. Therefore, Cavalitide binds much more strongly to the GLP1 receptor compared to semaglutide. The advantages of this include a higher potency and an outcompetition compared to native GLP1. However, disadvantages include slower dissociation and a risk of overactivation causing increased side effects like nausea.

## Task 2: Simulate the pharmacokinetics of Cavalitide treatment

Your boss is really excited about the progress of your team in Task 1 on the development of Cavalitide. You’re now assigned to help design the Phase 1 clinical trial for the pill version of Cavalitide. 

1. Review the example code for modeling semaglutide pharmacokinetics (semaglutidePK.py). Run simulations for both Ozempic (subcutaneous injection once per week) and Wegovy (daily pill).  Replace the portion of the code using an explicit Euler’s method with use of the more powerful built-in function scipy.integrate.solve_ivp. Rerun the code to create plots that compare the predicted profiles of Ozempic and Wegovy.
2. Now, adapt semaglutidePK.py to additionally incorporate simulations of Cavalitide (daily pill), which has an increased absorption rate ka = 0.006 1/h. Plot how Cavalitide blood concentration changes over time.
3. Given the higher absorption rate, you hypothesize that you can further optimize the dosing strategy. Adjust one or two drug or dosing specific parameters until you achieve a steady state blood concentration close to 0.5 nM. Plot your result. What did you change about the dosing strategy and why?

## Task 3: Simulate a combined pharmacokinetics-pharmacokinetics (PKPD) model that predicts GLP1 receptor activity during treatment

The Phase 1 clinical trial was a success, and you achieved the desired 0.5 nM concentration of Cavalitide in your patients. Now it’s time to further customize the dosing for the Phase 2 clinical trial, where you’re more interested in efficacy. You’ll need to combine the PK model (Task 2) and PD model (Task 1) to predict how the fraction of activated receptors changes over time.

1. Embed the ligand-receptor model of Cavalitide (from Task 1, but just using your optimized Kd) into your cavalitidePK.py model. Let’s call this combined model cavalitidePKPD.py. Predict how the fraction of active GLP1R receptors changes over the first 100 days of treatment. 
2. Further optimize your dosing strategy so that you achieve close to 90% active GLP1 receptors at steady state (around 100 days).  Which dosing parameters did you adjust? Provide an interpretation of the clinical benefits of this adjustment.


## Conclusions:

Explain what your analyses suggest about the potential for Cavalitide in competition with semaglutide (oral or injected).

## Limitations and Future Work:

Describe the limitations of your project. If you had more time or additional data, explain how you would improve the analysis.

There are several limitations in our project, one of which is the amount of experimental data we have in the Cavalitide data set. In task 2 particularly, we assumed the oral bioavailability of Cavalitide because we did not have experimental data. We also made the assumption that the drug has constant absporption and elimination, which may affect the accuracy of our results.

In our future work we would collect and use measured data for absportion and elimination rates, which would increase the accuracy of our results and reduce the possibility of error due to our assumption.

## AI Usage Statement:
*This is where you will provide a short description on whether AI (and which AI) was used for any of the sections above and whether it helped or why it did not help.*
- *Example statement: "We used Claude Code to generate our linear regression plot. At first, it did not include an R^2 calculation, so we added it to the code afterwards."*

## GitHub Repository Link:
*Please provide the link to your shared GitHub repository that includes this notebook and any other scripts, notebooks, or data that you used for this project.*


## References:
Mayo Clinic. (2023, July 22). Obesity. Mayo Clinic. https://www.mayoclinic.org/diseases-conditions/obesity/symptoms-causes/syc-20375742

## NOTES FROM YOUR TEAM: 
*This is where our team is taking notes and recording activity.*

## QUESTIONS FOR YOUR TA: 
*These are questions we have for our TA.*